In [2]:
import pandas as pd
import os
from dotenv import load_dotenv
from pathlib import Path
from sklearn.model_selection import StratifiedKFold

load_dotenv()
data_pth = Path(os.getenv("DATA_PATH", "data"))
train_csv_pth = f"{data_pth}/train.csv"
train_img_pth = Path(f"{data_pth}/train_images")

train_csv = pd.read_csv(train_csv_pth)
train_csv = train_csv.copy()

In [ ]:
imageide = train_csv["ImageId"].value_counts()
duplicates_combined = imageide.index.tolist()
train_file_name = [
    file_path.name for file_path in train_img_pth.iterdir() if file_path.is_file()
]

# Grouped defect profiles
class_ = []
for item in duplicates_combined:
    duplicate_statuses = train_csv[["ImageId", "ClassId"]].loc[
        train_csv["ImageId"] == item
    ]
    class_.append((item, "&".join(map(str, duplicate_statuses.ClassId.tolist()))))

# Matched train.csv with train_images and assigned "Clean" profile to train_images without defects
for obj in train_file_name:
    if obj not in duplicates_combined:
        class_.append((obj, "Clean"))

In [17]:
len(train_file_name) == len(class_)

True

In [6]:
import csv

headers = ["ImageId", "Class"]
file_pth = Path(os.getenv("CURRENT_FILE_PATH", ""))
with open("train_folds.csv", mode="w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(headers)
    writer.writerows(class_)

In [8]:
train_fold = pd.read_csv("train_folds.csv")
train_fold["Fold"] = -1

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for fold, (train_idx, val_idx) in enumerate(skf.split(train_fold, train_fold["Class"])):
    train_fold.loc[val_idx, "Fold"] = fold

train_fold.to_csv("train_folds.csv", index=False)

c:\Users\Chinedu\Documents\segmentation_project\.venv\Lib\site-packages\sklearn\model_selection\_split.py:812: UserWarning: The least populated class in y has only 1 members, which is less than n_splits=5.
  warnings.warn(


In [ ]:
def split_df_on_fold(
    val_fold_no: int,
    fold_df: pd.DataFrame,
    train_csv_df: pd.DataFrame,
    train_img_pth: Path,
):
    # Get all folds based on val_fold_no
    val_idx = fold_df.loc[fold_df["Folds"] == val_fold_no]
    image_id = val_idx.ImageId.tolist()  # Image Id in list

    for id in image_id:
        if id in train_csv_df["ClassId"]:
            val = train_csv_df.loc[id, :]
        

In [22]:
train_csv.ImageId.dtype

<StringDtype(storage='python', na_value=nan)>

In [28]:
train_fold.ImageId.dtype

<StringDtype(storage='python', na_value=nan)>

In [23]:
udy = "soft"
type(udy)

str

In [ ]:
val_idx = train_fold.loc[train_fold["Fold"] == 1]

for id in val_idx["ImageId"]:
    val = train_csv.loc[train_csv["ImageId"] == id]
    print(val)



            ImageId  ClassId                                      EncodedPixels
6101  db4867ee8.jpg        1  349941 2 350194 6 350447 11 350700 15 350953 1...
6102  db4867ee8.jpg        2  354411 17 354634 50 354857 82 355096 99 355351...
6103  db4867ee8.jpg        3                              233729 3008 236801 64
          ImageId  ClassId                                      EncodedPixels
18  008ef3d74.jpg        1  356336 4 356587 11 356838 18 357089 25 357340 ...
19  008ef3d74.jpg        2  375439 5 375687 14 375935 24 376182 34 376430 ...
          ImageId  ClassId                                      EncodedPixels
26  00c88fed0.jpg        1  10474 7 10728 15 10983 18 11239 21 11494 24 11...
27  00c88fed0.jpg        2  13428 8 13684 24 13940 39 14196 55 14452 71 14...
           ImageId  ClassId                                      EncodedPixels
108  03db6bbc3.jpg        3  29096 5 29342 15 29588 25 29835 34 30081 44 30...
109  03db6bbc3.jpg        4  82462 2 82715 7 82969 11 

In [35]:
val

,ImageId,ClassId,EncodedPixels
